# 06 — For Each Loops

> **DataMindAI | Ahmed**

## Why For-Each?
Run the **same task N times** with different inputs — without creating N separate tasks.

## Scenario
Process assessment data for each course in parallel:
- Data Science assessment
- Engineering assessment
- Mathematics assessment

```
generate_course_list
         │
         ▼
  [For-Each: process_each_course]
  ┌───────────────────────────────┐
  │  Iteration 1: Data Science    │
  │  Iteration 2: Engineering     │  (all 3 run simultaneously)
  │  Iteration 3: Mathematics     │
  └───────────────────────────────┘
```

## Job UI Setup
1. Task 1: `generate_course_list` — publishes `course_list` as a JSON array
2. Task 2: **For Each** type — name: `process_each_course`
   - Inputs: `{{tasks.generate_course_list.values.course_list}}`
   - Concurrency: 3
3. Nested task: `process_single_course`
   - Parameter: `input: {{input}}`

> **Concurrency tip:** Set between 10–20 for production. Higher = faster but more compute.

---
## Step 1 — Generate the course list
---

In [ ]:
# ── TASK 1: generate_course_list ─────────────────────────────────────────────
# Builds the array that the For-Each task will iterate over.
# Each dict in the list becomes one iteration.
import json

# Method 1: Hard-coded list (simple demo)
course_list = [
    {'course_id': 'C001', 'course_name': 'Data Science', 'pass_mark': 40, 'department': 'Computing'},
    {'course_id': 'C002', 'course_name': 'Engineering',  'pass_mark': 40, 'department': 'Technology'},
    {'course_id': 'C003', 'course_name': 'Mathematics',  'pass_mark': 40, 'department': 'Science'},
]

print(f'Generated {len(course_list)} courses for parallel processing:')
for c in course_list:
    print(f'  {c["course_id"]} — {c["course_name"]} (dept: {c["department"]})')

# Publish as task value for For-Each to consume
dbutils.jobs.taskValues.set(key='course_list', value=json.dumps(course_list))
print()
print('course_list published. For-Each will create one iteration per course.')

In [ ]:
# ── Method 2: Query from SQL mapping table (production pattern) ───────────────
# In production, read the list from a config table rather than hard-coding it.
# This way, adding a new course requires only a DB insert — no code change.
import json

df_courses = spark.table('demo_catalog.config.courses')

course_list_from_db = [
    {'course_id': r['course_id'], 'course_name': r['course_name'],
     'department': r['department'], 'pass_mark': r['pass_mark']}
    for r in df_courses.collect()
]

print(f'Loaded {len(course_list_from_db)} courses from config.courses table:')
print(json.dumps(course_list_from_db, indent=2))

# Same publish — but now driven by the database
dbutils.jobs.taskValues.set(key='course_list', value=json.dumps(course_list_from_db))
print()
print('Advantage: add a row to config.courses, get a new iteration automatically.')

---
## Step 2 — Nested task (runs once per course)
---

In [ ]:
# ── NESTED TASK: process_single_course ───────────────────────────────────────
# This notebook runs ONCE PER COURSE in the For-Each loop.
# Receives the current iteration via widget parameter 'input'.
#
# For-Each nested task parameter config in Job UI:
#   Key: input  |  Value: {{input}}
import json

# In a job: receives the full dict for this iteration
try:
    raw = dbutils.widgets.get('input')
    params = json.loads(raw)
except:
    # Local fallback for interactive testing
    params = {'course_id': 'C001', 'course_name': 'Data Science',
              'pass_mark': 40, 'department': 'Computing'}

course_id   = params['course_id']
course_name = params['course_name']
pass_mark   = params['pass_mark']
department  = params['department']

print(f'Processing course: {course_name} ({course_id})')
print(f'Department: {department}  |  Pass mark: {pass_mark}%')

# Filter students for this course only
from pyspark.sql.functions import when, col, avg, count, round as spark_round, lit

df = spark.table('demo_catalog.bronze.students_raw')
df_course = df.filter(f"course = '{course_name}'")

total     = df_course.count()
passed    = df_course.filter(f'score >= {pass_mark}').count()
failed    = total - passed
pass_rate = round((passed / total) * 100, 1) if total > 0 else 0
avg_score = round(df_course.agg(avg('score')).collect()[0][0] or 0, 1)
at_risk   = df_course.filter('score < 40 OR attendance < 50').count()

print(f'Students: {total}  |  Pass: {passed}  |  Fail: {failed}')
print(f'Pass rate: {pass_rate}%  |  Avg score: {avg_score}')
print(f'At-risk: {at_risk}')

# Write course-specific summary
from pyspark.sql import Row
import datetime

spark.createDataFrame([Row(
    course_id=course_id, course_name=course_name, department=department,
    total_students=total, pass_count=passed, fail_count=failed,
    pass_rate=pass_rate, avg_score=avg_score, at_risk_count=at_risk,
    report_date=datetime.date.today().isoformat()
)]).write.format('delta').mode('append').saveAsTable('demo_catalog.gold.course_summaries')

print(f'Summary written for {course_name}.')

---
## Simulate running all three iterations locally
---

In [ ]:
# ── Simulate the For-Each loop running all 3 courses ─────────────────────────
# In the real job, each of these runs as a separate parallel task.
# Here we simulate them sequentially to see the full output.
import json

courses = [
    {'course_id': 'C001', 'course_name': 'Data Science', 'pass_mark': 40, 'department': 'Computing'},
    {'course_id': 'C002', 'course_name': 'Engineering',  'pass_mark': 40, 'department': 'Technology'},
    {'course_id': 'C003', 'course_name': 'Mathematics',  'pass_mark': 40, 'department': 'Science'},
]

from pyspark.sql.functions import avg, col
from pyspark.sql import Row
import datetime

df_all = spark.table('demo_catalog.bronze.students_raw')
results = []

for params in courses:
    course_name = params['course_name']
    df_c = df_all.filter(f"course = '{course_name}'")
    total     = df_c.count()
    passed    = df_c.filter('score >= 40').count()
    avg_sc    = round(df_c.agg(avg('score')).collect()[0][0] or 0, 1)
    pass_rate = round((passed/total)*100,1) if total > 0 else 0
    results.append(Row(
        course=course_name, students=total,
        passed=passed, pass_rate=pass_rate, avg_score=avg_sc,
        report_date=datetime.date.today().isoformat()
    ))
    print(f'[Iteration] {course_name:15s} | {total} students | Pass: {pass_rate}% | Avg: {avg_sc}')

spark.createDataFrame(results).write.format('delta').mode('overwrite').saveAsTable('demo_catalog.gold.course_summaries')
print()
print('All iterations complete. Gold table written:')
spark.table('demo_catalog.gold.course_summaries').show()